# Sequence Hotspot Finder v1.0 Colab UI
GitHub repository를 clone해서 실행하는 editable UI입니다. `REPO_URL`만 본인 주소로 바꾸세요.

In [ ]:

# =========================================================
# Sequence Hotspot Finder v1.0 — GitHub Clone + Editable Colab UI
# =========================================================
# 1. REPO_URL을 본인 GitHub 주소로 바꾸세요.
# 2. Runtime → Change runtime type → GPU 권장.
# 3. 이 셀을 실행하면 탭형 UI가 나타납니다.
# =========================================================

REPO_URL = "https://github.com/YOUR_ID/sequence-hotspot-finder.git"
BRANCH = "main"

import os, sys, json, shutil, subprocess
from pathlib import Path

WORKDIR = Path("/content/sequence-hotspot-finder")
RUNTIME_DIR = Path("/content/seqhotspot_runtime")
OUTPUT_DIR = RUNTIME_DIR / "outputs"

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
print("Cloning repository...")
subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))

print("Installing requirements...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from sequence_hotspot_finder.engine import analyze_input, load_config
from sequence_hotspot_finder.validation import validate_token_db, validate_sidechain_mod_db, validate_config, validate_domain_csv, validate_position_csv

RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOKEN_DB_PATH = RUNTIME_DIR / "token_db_runtime.csv"
SIDECHAIN_DB_PATH = RUNTIME_DIR / "sidechain_mod_db_runtime.csv"
CONFIG_PATH = RUNTIME_DIR / "config_runtime.json"
DOMAIN_PATH = RUNTIME_DIR / "domains_runtime.csv"
STRUCTURE_PATH = RUNTIME_DIR / "structure_runtime.csv"
CONSERVATION_PATH = RUNTIME_DIR / "conservation_runtime.csv"

for src, dst in [
    (WORKDIR/"data/token_db.csv", TOKEN_DB_PATH),
    (WORKDIR/"data/sidechain_mod_db.csv", SIDECHAIN_DB_PATH),
    (WORKDIR/"data/default_config.json", CONFIG_PATH),
    (WORKDIR/"examples/example_domains.csv", DOMAIN_PATH),
    (WORKDIR/"examples/example_structure_features.csv", STRUCTURE_PATH),
    (WORKDIR/"examples/example_conservation_features.csv", CONSERVATION_PATH),
]:
    if src.exists(): shutil.copy(src, dst)

example_input = (WORKDIR/"examples/example_input.fasta").read_text(encoding="utf-8")

def read_text(p): return Path(p).read_text(encoding="utf-8")
def write_text(p, t): Path(p).write_text(str(t), encoding="utf-8")
def preview_csv(text, rows=10):
    from io import StringIO
    df = pd.read_csv(StringIO(text))
    display(df.head(rows))
    print(f"Rows={len(df)}, Columns={len(df.columns)}")

def plot_top(df, top_n=40):
    d = df[df.hotspot_score.notna()].sort_values("hotspot_score", ascending=False).head(top_n)
    if d.empty:
        print("No scored residues."); return
    labels = [f"{int(r.display_position)}:{r.input_token}" for _, r in d.iterrows()]
    plt.figure(figsize=(max(10, len(labels)*0.35), 4))
    plt.bar(range(len(labels)), d.hotspot_score.astype(float))
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.ylim(0,1); plt.ylabel("hotspot_score"); plt.title("Top candidate residues")
    plt.tight_layout(); plt.show()

# ---------------- UI ----------------
sequence_box = widgets.Textarea(value=example_input, description="Sequence", layout=widgets.Layout(width="100%", height="260px"))
use_esm = widgets.Checkbox(value=False, description="Use ESM-2")
esm_model = widgets.Dropdown(options=["esm2_t6_8M_UR50D", "esm2_t12_35M_UR50D", "esm2_t30_150M_UR50D"], value="esm2_t6_8M_UR50D", description="Model")
window = widgets.IntText(value=900, description="Window")
overlap = widgets.IntText(value=150, description="Overlap")
batch = widgets.IntSlider(value=4, min=1, max=16, description="Batch")
masked = widgets.Checkbox(value=True, description="Masked marginal")
mut = widgets.Checkbox(value=True, description="Mutation sensitivity")
topn = widgets.IntSlider(value=30, min=5, max=200, step=5, description="Top N")
merge_mode = widgets.Dropdown(options=["model_position", "display_position", "original_position", "residue_position"], value="model_position", description="Merge pos")

token_box = widgets.Textarea(value=read_text(TOKEN_DB_PATH), description="token_db.csv", layout=widgets.Layout(width="100%", height="320px"))
sidechain_box = widgets.Textarea(value=read_text(SIDECHAIN_DB_PATH), description="sidechain.csv", layout=widgets.Layout(width="100%", height="220px"))
config_box = widgets.Textarea(value=read_text(CONFIG_PATH), description="config.json", layout=widgets.Layout(width="100%", height="320px"))
domain_box = widgets.Textarea(value=read_text(DOMAIN_PATH), description="domains.csv", layout=widgets.Layout(width="100%", height="150px"))
structure_box = widgets.Textarea(value=read_text(STRUCTURE_PATH), description="structure.csv", layout=widgets.Layout(width="100%", height="150px"))
conservation_box = widgets.Textarea(value=read_text(CONSERVATION_PATH), description="conserv.csv", layout=widgets.Layout(width="100%", height="150px"))
use_domain = widgets.Checkbox(value=False, description="Use domain CSV")
use_structure = widgets.Checkbox(value=False, description="Use structure CSV")
use_conservation = widgets.Checkbox(value=False, description="Use conservation CSV")

validate_btn = widgets.Button(description="Validate inputs", button_style="info")
run_btn = widgets.Button(description="Run analysis", button_style="success")
download_btn = widgets.Button(description="Download ZIP", button_style="primary")
out = widgets.Output()
latest = {"zip": None}

def build_config():
    cfg = json.loads(config_box.value)
    cfg.update({
        "use_esm": bool(use_esm.value), "esm_model": esm_model.value,
        "window_size": int(window.value), "overlap": int(overlap.value),
        "batch_size": int(batch.value), "use_masked_marginal": bool(masked.value),
        "use_mutation_sensitivity": bool(mut.value), "top_n": int(topn.value),
        "merge_position_mode": merge_mode.value,
    })
    return cfg

def save_runtime():
    write_text(TOKEN_DB_PATH, token_box.value)
    write_text(SIDECHAIN_DB_PATH, sidechain_box.value)
    cfg = build_config(); config_box.value = json.dumps(cfg, indent=2, ensure_ascii=False); write_text(CONFIG_PATH, config_box.value)
    write_text(DOMAIN_PATH, domain_box.value); write_text(STRUCTURE_PATH, structure_box.value); write_text(CONSERVATION_PATH, conservation_box.value)
    return cfg

def on_validate(_):
    with out:
        clear_output()
        try:
            save_runtime()
            validate_token_db(pd.read_csv(TOKEN_DB_PATH))
            validate_sidechain_mod_db(pd.read_csv(SIDECHAIN_DB_PATH))
            validate_config(json.loads(config_box.value))
            if use_domain.value: validate_domain_csv(pd.read_csv(DOMAIN_PATH))
            if use_structure.value: validate_position_csv(pd.read_csv(STRUCTURE_PATH), "structure")
            if use_conservation.value: validate_position_csv(pd.read_csv(CONSERVATION_PATH), "conservation")
            print("Validation passed.")
        except Exception as e:
            print("Validation failed:", type(e).__name__, e)

def on_run(_):
    with out:
        clear_output()
        try:
            cfg = save_runtime()
            print("Running analysis...")
            res = analyze_input(
                sequence_box.value, config=cfg, token_db_path=TOKEN_DB_PATH, sidechain_mod_db_path=SIDECHAIN_DB_PATH,
                outdir=OUTPUT_DIR,
                domains_path=DOMAIN_PATH if use_domain.value else None,
                structure_features_path=STRUCTURE_PATH if use_structure.value else None,
                conservation_features_path=CONSERVATION_PATH if use_conservation.value else None,
            )
            latest["zip"] = res["zip_path"]
            print("Done.")
            print("Full CSV:", res["full_csv"])
            print("Top CSV :", res["top_csv"])
            print("ZIP     :", res["zip_path"])
            display(res["top_df"].head(int(topn.value)))
            plot_top(res["full_df"], top_n=min(int(topn.value), 60))
        except Exception as e:
            print("Run failed:", type(e).__name__, e)

def on_download(_):
    with out:
        if latest["zip"] and Path(latest["zip"]).exists():
            files.download(latest["zip"])
        else:
            print("No ZIP yet. Run analysis first.")

validate_btn.on_click(on_validate)
run_btn.on_click(on_run)
download_btn.on_click(on_download)

input_tab = widgets.VBox([widgets.HTML("<b>Paste FASTA, direct sequence, or modified peptide.</b>"), sequence_box])
engine_tab = widgets.VBox([use_esm, esm_model, widgets.HBox([window, overlap, batch]), widgets.HBox([masked, mut]), widgets.HBox([topn, merge_mode])])
token_tab = widgets.VBox([widgets.HTML("<b>Edit token_db.csv</b>"), token_box, widgets.HTML("<b>Edit sidechain_mod_db.csv</b>"), sidechain_box])
config_tab = widgets.VBox([config_box])
optional_tab = widgets.VBox([use_domain, domain_box, use_structure, structure_box, use_conservation, conservation_box])
run_tab = widgets.VBox([widgets.HBox([validate_btn, run_btn, download_btn]), out])

tabs = widgets.Tab(children=[input_tab, engine_tab, token_tab, config_tab, optional_tab, run_tab])
for i, title in enumerate(["Input", "Engine", "Token DB", "Config", "Optional CSV", "Run"]): tabs.set_title(i, title)
display(widgets.HTML("<h2>Sequence Hotspot Finder v1.0</h2><p>GitHub clone/import 방식의 editable Colab UI입니다.</p>"))
display(tabs)
print("Ready. Runtime directory:", RUNTIME_DIR)
